<a href="https://colab.research.google.com/github/kaykesm/MLCB_CK_2026/blob/main/C%C3%B3digoAtividade01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# ATIVIDADE 1 - CHATBOT VERSÃO 1 (KNN)
# SAC MÓVEIS RESIDENCIAIS
# ==============================================================================

import numpy as np
import pandas as pd
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# ==============================================================================
# 1. GERANDO O DATASET
# ==============================================================================

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': [
            'quero comprar',
            'qual o preco do',
            'tem cupom para',
            'como faco para adquirir',
            'desejo orcamento de'
        ],
        'o': [
            'sofa retratil 3 lugares',
            'conjunto de mesa de jantar',
            'guarda roupa casal',
            'painel para tv',
            'colchao queen size'
        ]
    },

    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': [
            'como montar o',
            'onde baixo o manual do',
            'estou com duvida no',
            'veio faltando parafuso no',
            'preciso de assistencia para'
        ],
        'o': [
            'armario de cozinha',
            'rack da sala',
            'berco do bebe',
            'esquema de montagem',
            'manual da estante'
        ]
    },

    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': [
            'preciso trocar o',
            'quero devolver a',
            'como solicito o estorno do',
            'desejo solicitar a troca da',
            'como funciona a devolucao do'
        ],
        'o': [
            'produto com defeito',
            'mesa que veio arranhada',
            'cadeira no prazo de 7 dias',
            'pedido cancelado',
            'item com avaria'
        ]
    },

    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': [
            'estou indignado com o',
            'quero fazer uma queixa do',
            'estou reclamando do',
            'produto veio quebrado e o',
            'atendimento horrivel do'
        ],
        'o': [
            'atraso na minha entrega',
            'servico de montagem',
            'sac que nao responde',
            'pos venda da loja',
            'estado do meu movel'
        ]
    },

    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': [
            'onde esta o meu',
            'qual o prazo de entrega do',
            'como rastreio a',
            'qual a transportadora do',
            'quando chega o'
        ],
        'o': [
            'meu pedido',
            'codigo de rastreamento',
            'movel comprado',
            'status do envio',
            'agendamento da entrega'
        ]
    }
}


amostras = []

random.seed(42)

for intencao, comp in templates.items():

    for _ in range(20):

        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])

        frase = f"{s} {a} {o}".strip().capitalize()

        amostras.append({
            'texto': frase,
            'intencao': intencao
        })


df_moveis = pd.DataFrame(amostras)

df_moveis.to_csv(
    'dataset_moveis_100.csv',
    index=False,
    encoding='utf-8'
)

print("Dataset criado com sucesso!")


# ==============================================================================
# 2. CARREGANDO O DATASET
# ==============================================================================

df = pd.read_csv('dataset_moveis_100.csv')

print("\nQuantidade de frases:", len(df))

print("\nQuantidade por intenção:")
print(df['intencao'].value_counts())


# ==============================================================================
# 3. DIVISÃO TREINO E TESTE
# ==============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.30,
    random_state=42,
    stratify=df['intencao']
)

print("\nQuantidade de dados para treinamento:", len(X_train))
print("Quantidade de dados para teste:", len(X_test))


# ==============================================================================
# 4. PIPELINE KNN
# ==============================================================================

pipeline_knn = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', KNeighborsClassifier(
        n_neighbors=3,
        metric='cosine'
    ))
])


# ==============================================================================
# 5. TREINAMENTO
# ==============================================================================

pipeline_knn.fit(X_train, y_train)

print("\nModelo KNN treinado com sucesso!")


# ==============================================================================
# 6. PREVISÕES
# ==============================================================================

y_pred = pipeline_knn.predict(X_test)


# ==============================================================================
# 7. CLASSIFICATION REPORT
# ==============================================================================

print("\n==============================================")
print("CLASSIFICATION REPORT - KNN")
print("==============================================")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ==============================================================================
# 8. MATRIZ DE CONFUSÃO
# ==============================================================================

print("\n==============================================")
print("MATRIZ DE CONFUSÃO - KNN")
print("==============================================")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ==============================================================================
# 9. TESTES MANUAIS
# ==============================================================================

LIMIAR_CONFIANCA = 0.50

print("\n==============================================")
print("BATERIA DE 10 TESTES")
print("==============================================")


for i in range(1, 11):

    print(f"\n[Teste {i}/10]")

    frase = input(
        "Digite a frase do cliente: "
    ).strip()

    probs = pipeline_knn.predict_proba([frase])[0]

    maior_prob = np.max(probs)

    intencao = pipeline_knn.predict([frase])[0]

    if maior_prob >= LIMIAR_CONFIANCA:

        print(f"Intenção identificada: {intencao}")
        print(f"Confiança: {maior_prob:.2%}")

    else:

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )

        print(
            f"Maior confiança encontrada: {maior_prob:.2%}"
        )


print("\n==============================================")
print("ATIVIDADE 1 FINALIZADA")
print("==============================================")


Dataset criado com sucesso!

Quantidade de frases: 100

Quantidade por intenção:
intencao
vendas                20
suporte               20
trocas_devolucoes     20
reclamacoes           20
logistica_entregas    20
Name: count, dtype: int64

Quantidade de dados para treinamento: 70
Quantidade de dados para teste: 30

Modelo KNN treinado com sucesso!

CLASSIFICATION REPORT - KNN
                    precision    recall  f1-score   support

logistica_entregas       1.00      1.00      1.00         6
       reclamacoes       1.00      1.00      1.00         6
           suporte       1.00      1.00      1.00         6
 trocas_devolucoes       1.00      1.00      1.00         6
            vendas       1.00      1.00      1.00         6

          accuracy                           1.00        30
         macro avg       1.00      1.00      1.00        30
      weighted avg       1.00      1.00      1.00        30


MATRIZ DE CONFUSÃO - KNN
[[6 0 0 0 0]
 [0 6 0 0 0]
 [0 0 6 0 0]
 [0 0 0 6 0